# Mediator Design Pattern 

explained using the classic Air Traffic Control (ATC) example.

#### The Concept

The Mediator Pattern restricts direct communications between objects and forces them to collaborate only via a mediator object.
- **Without Mediator**: Pilot A talks to Pilot B, Pilot B talks to Pilot C... chaos.
- **With Mediator**: Pilot A talks to the Tower. The Tower talks to Pilot B.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we define an `Interface` for the Mediator so that colleagues (Planes) are not coupled to the specific Tower implementation. We also use an abstract `BaseComponent` for the Planes to hold the reference to the Mediator.

#### MEDIATOR INTERFACE

In [1]:
from abc import ABC, abstractmethod

class IAirTrafficControl(ABC):
    @abstractmethod
    def notify(self, sender: 'BaseFlight', event: str):
        pass

#### ABSTRACT COLLEAGUE (The Plane Base)

In [2]:
class BaseFlight:
    def __init__(self, atc: IAirTrafficControl):
        self.atc = atc # Reference to the Mediator

    def send_notification(self, event: str):
        self.atc.notify(self, event)

#### CONCRETE COLLEAGUES (Specific Planes)

In [3]:
class Boeing747(BaseFlight):
    def request_landing(self):
        print("Boeing 747: Requesting landing clearance.")
        self.send_notification("landing_request")

    def land(self):
        print("Boeing 747: Landing wheels deployed. Touchdown.")

    def permit_granted(self):
        print("Boeing 747: Clearance received. Approaching runway.")

class AirbusA320(BaseFlight):
    def request_takeoff(self):
        print("Airbus A320: Requesting takeoff.")
        self.send_notification("takeoff_request")

    def take_off(self):
        print("Airbus A320: Taking off into the sky.")

    def halt(self):
        print("Airbus A320: Holding position on tarmac.")

#### CONCRETE MEDIATOR (The Logic Hub)

In [4]:
class ControlTower(IAirTrafficControl):
    def __init__(self):
        # The Mediator needs to know about the specific components
        self.boeing = None
        self.airbus = None

    def register(self, boeing: Boeing747, airbus: AirbusA320):
        self.boeing = boeing
        self.airbus = airbus

    def notify(self, sender: BaseFlight, event: str):
        # This is where the spaghetti logic lives (centralized)
        if event == "landing_request":
            if isinstance(sender, Boeing747):
                # Logic: If Boeing wants to land, stop the Airbus
                print("[Tower]: Boeing requesting landing. Freezing runway.")
                self.airbus.halt()
                self.boeing.permit_granted()
        
        elif event == "takeoff_request":
            # Logic: If Airbus wants to take off, check runway...
            print("[Tower]: Airbus requesting takeoff. Granting...")
            self.airbus.take_off()

#### CLIENT CODE

In [5]:
def main():
    tower = ControlTower()

    # Create planes and link them to the tower
    flight1 = Boeing747(tower)
    flight2 = AirbusA320(tower)

    # Register them so the tower knows who they are
    tower.register(flight1, flight2)

    # Trigger interactions
    print("--- Interaction 1 ---")
    flight1.request_landing()

if __name__ == "__main__":
    main()

--- Interaction 1 ---
Boeing 747: Requesting landing clearance.
[Tower]: Boeing requesting landing. Freezing runway.
Airbus A320: Holding position on tarmac.
Boeing 747: Clearance received. Approaching runway.


## The Pythonic Way

#### In Python, we can simplify this:

- **No Abstract Base Class**: We don't need `BaseFlight` just to hold a variable.
- **Dynamic Registration**: The Mediator can just hold a generic list or dictionary of objects.
- **Loose Coupling**: The "Planes" don't necessarily need to inherit from anything. They just need to know who the mediator is.

Here is a version representing a **Chat Room Mediator**, where the Mediator is simply a central registry that broadcasts messages.

#### THE COLLEAGUES (Simple Classes)

In [6]:
from dataclasses import dataclass

@dataclass
class User:
    name: str
    mediator: 'ChatRoom' = None # Type hint only

    def send(self, message: str):
        print(f"[{self.name} sends]: {message}")
        # Delegate directly to the mediator
        if self.mediator:
            self.mediator.broadcast(self, message)

    def receive(self, sender_name: str, message: str):
        print(f"   -> {self.name} received from {sender_name}: '{message}'")

#### THE PYTHONIC MEDIATOR

In [7]:
class ChatRoom:
    """
    Acts as the Mediator.
    """
    def __init__(self):
        self.users = []

    def join(self, user: User):
        self.users.append(user)
        user.mediator = self # Automatically link the user to this room
        print(f"[Room]: {user.name} joined the chat.")

    def broadcast(self, sender: User, message: str):
        # Logic: Send to everyone EXCEPT the sender
        for user in self.users:
            if user != sender:
                user.receive(sender.name, message)

#### CLIENT CODE

In [8]:
def main():
    # 1. Create Mediator
    dev_channel = ChatRoom()

    # 2. Create Users
    alice = User("Alice")
    bob = User("Bob")
    charlie = User("Charlie")

    # 3. Join Room (Mediator Setup)
    dev_channel.join(alice)
    dev_channel.join(bob)
    dev_channel.join(charlie)
    print()

    # 4. Interaction
    # Alice sends a message. She doesn't know Bob or Charlie exist.
    # She only knows the ChatRoom.
    alice.send("Hello everyone!")
    print()
    
    bob.send("Hey Alice!")

if __name__ == "__main__":
    main()

[Room]: Alice joined the chat.
[Room]: Bob joined the chat.
[Room]: Charlie joined the chat.

[Alice sends]: Hello everyone!
   -> Bob received from Alice: 'Hello everyone!'
   -> Charlie received from Alice: 'Hello everyone!'

[Bob sends]: Hey Alice!
   -> Alice received from Bob: 'Hey Alice!'
   -> Charlie received from Bob: 'Hey Alice!'


#### Key Differences

| Feature        | Classic OOP                                                        | Pythonic                                                         |
|----------------|--------------------------------------------------------------------|------------------------------------------------------------------|
| **Structure**  | Strict interfaces (`IMediator`) and abstract parents (`BaseColleague`). | Duck typing or simple classes.                                  |
| **Logic**      | Often uses large `if/else` blocks with `isinstance` checks.        | Uses lists, broadcasting, or dynamic dispatch.                  |
| **Coupling**   | Tighter coupling with explicit registration for specific classes.  | Looser coupling with a generic `join` method accepting any object with a `receive` method. |


#### When to use Mediator?

Use it when you have a **"Mesh"** of dependencies (Everything talks to everything) and you want to turn it into a **"Star"** topology (Everything talks to the Center). This typically happens in GUI forms (Checkbox A affects Dropdown B, which affects Button C) or Chat systems.

# Mediator Design Pattern 

explained using a complex, real-world example: **A Smart Home Automation Hub**.

#### The Scenario: The "Morning Routine" & "Security" Conflict

Imagine you have various smart devices:
- **Motion Sensor** (Porch)
- **Smart Bulb** (Living Room)
- **Smart Speaker** (Alexa/Google)
- **Security Alarm**

#### The Complexity:

- If motion is detected **during the day**, nothing happens.
- If motion is detected **at night**, turn on the lights.
- HOWEVER, if the **Alarm is Armed**, do not turn on the lights; instead, sound the siren and notify the police.

**Without Mediator**: The Motion Sensor would need references to the Light, the Clock, and the Alarm to check their states. It would become a "God Object" with massive if/else chains. 

**With Mediator (Hub)**: The Sensor just says "Motion Detected". The Hub decides what to do based on the state of the other devices.

## The Classic OOP Way (Java-Style)

We use a strict Interface for the Mediator and an Abstract Class for the Devices ("Colleagues"). The logic is centralized in the `ConcreteMediator`.

#### MEDIATOR INTERFACE

In [9]:
from abc import ABC, abstractmethod

class SmartHomeHub(ABC):
    @abstractmethod
    def notify(self, sender: object, event: str):
        pass

#### ABSTRACT COLLEAGUE (Device)

In [10]:
from abc import ABC

class SmartDevice(ABC):
    def __init__(self, hub: SmartHomeHub, name: str):
        self.hub = hub
        self.name = name

#### CONCRETE DEVICES (Dumb Objects)

> Note: These devices know NOTHING about each other.

In [11]:
class MotionSensor(SmartDevice):
    def trigger(self):
        print(f"[{self.name}] Motion detected! Sending signal to Hub...")
        self.hub.notify(self, "motion_detected")

class SmartBulb(SmartDevice):
    def turn_on(self):
        print(f"[{self.name}] Light is now ON.")

    def turn_off(self):
        print(f"[{self.name}] Light is now OFF.")

class SecurityAlarm(SmartDevice):
    def __init__(self, hub: SmartHomeHub, name: str):
        super().__init__(hub, name)
        self.is_armed = False

    def arm(self):
        print(f"[{self.name}] System ARMED.")
        self.is_armed = True

    def disarm(self):
        print(f"[{self.name}] System DISARMED.")
        self.is_armed = False

    def sound_siren(self):
        print(f"[{self.name}] 🚨 WEE-WOO! WEE-WOO! INTRUDER ALERT! 🚨")

#### CONCRETE MEDIATOR (The Brain)

In [13]:
class ConcreteHomeHub(SmartHomeHub):
    def __init__(self):
        # The Hub holds references to all devices
        self.sensor = None
        self.bulb = None
        self.alarm = None
        self.is_night = False # System State

    # Setup method to link devices
    def set_devices(self, sensor, bulb, alarm):
        self.sensor = sensor
        self.bulb = bulb
        self.alarm = alarm

    def set_time_of_day(self, is_night: bool):
        self.is_night = is_night
        print(f"[Hub] Time set to: {'Night' if is_night else 'Day'}")

    def notify(self, sender: object, event: str):
        # --- CENTRALIZED COMPLEX LOGIC ---
        
        if event == "motion_detected":
            if self.alarm.is_armed:
                # CRITICAL SECURITY RULE
                print("[Hub] Security Breach! Triggering Alarm.")
                self.alarm.sound_siren()
                # We do NOT turn on lights to keep element of surprise (or whatever logic)
            
            elif self.is_night:
                # CONVENIENCE RULE
                print("[Hub] It's dark. Turning on lights for convenience.")
                self.bulb.turn_on()
            
            else:
                # DEFAULT RULE
                print("[Hub] It's daytime. Ignoring motion.")

#### CLIENT CODE

In [14]:
def main():
    # 1. Create the Mediator
    hub = ConcreteHomeHub()

    # 2. Create Devices (Injecting Hub)
    sensor = MotionSensor(hub, "Porch Sensor")
    bulb = SmartBulb(hub, "Living Room Light")
    alarm = SecurityAlarm(hub, "Home Security")

    # 3. Link them up
    hub.set_devices(sensor, bulb, alarm)

    print("--- SCENARIO 1: Day time, Alarm Off ---")
    hub.set_time_of_day(is_night=False)
    sensor.trigger() # Should do nothing

    print("\n--- SCENARIO 2: Night time, Alarm Off ---")
    hub.set_time_of_day(is_night=True)
    sensor.trigger() # Should turn on lights

    print("\n--- SCENARIO 3: Night time, Alarm ARMED ---")
    alarm.arm()
    sensor.trigger() # Should sound siren, NOT lights

if __name__ == "__main__":
    main()

--- SCENARIO 1: Day time, Alarm Off ---
[Hub] Time set to: Day
[Porch Sensor] Motion detected! Sending signal to Hub...
[Hub] It's daytime. Ignoring motion.

--- SCENARIO 2: Night time, Alarm Off ---
[Hub] Time set to: Night
[Porch Sensor] Motion detected! Sending signal to Hub...
[Hub] It's dark. Turning on lights for convenience.
[Living Room Light] Light is now ON.

--- SCENARIO 3: Night time, Alarm ARMED ---
[Home Security] System ARMED.
[Porch Sensor] Motion detected! Sending signal to Hub...
[Hub] Security Breach! Triggering Alarm.
[Home Security] 🚨 WEE-WOO! WEE-WOO! INTRUDER ALERT! 🚨


## The Pythonic Way

In Python, forcing every device to inherit from a base class and hold a reference to a "Hub" object can be rigid. Instead, we can use a **Central Controller** that wires interactions using **functions (callbacks)** or specific event handlers. The devices can remain completely independent (standalone)—they don't even need to know a Hub exists.

This is often called the **"Signals and Slots"** approach (popular in Qt/PyQt) or simply Event Wiring.

#### STANDALONE DEVICES (No Hub Reference!)

In [16]:
from dataclasses import dataclass

class MotionSensor:
    def __init__(self, name):
        self.name = name
        self.on_motion_detected = [] # List of callbacks (Pythonic Event)

    def trigger(self):
        print(f"[{self.name}] Motion detected!")
        # Fire all attached callbacks
        for callback in self.on_motion_detected:
            callback()

class SmartBulb:
    def __init__(self, name):
        self.name = name

    def turn_on(self): print(f"[{self.name}] ON")
    def turn_off(self): print(f"[{self.name}] OFF")

class AlarmSystem:
    def __init__(self):
        self.armed = False

    def toggle_arm(self, status): 
        self.armed = status
        print(f"[Alarm] Armed: {self.armed}")

    def sound_siren(self):
        print(f"[Alarm] 🚨 SIREN SOUNDING 🚨")

#### THE MEDIATOR (Automation Controller)

In [17]:
class HomeAutomationController:
    """
    This class owns the logic. It observes the devices
    and manipulates them.
    """
    def __init__(self, sensor, bulb, alarm):
        self.sensor = sensor
        self.bulb = bulb
        self.alarm = alarm
        self.is_night_mode = False

        # WIRING: We attach our logic to the sensor's event list
        # The sensor doesn't know 'who' this function belongs to.
        self.sensor.on_motion_detected.append(self._handle_motion)

    def set_night_mode(self, status):
        self.is_night_mode = status
        print(f"[Controller] Night Mode: {status}")

    def _handle_motion(self):
        """
        The Logic Core. This function runs when sensor fires.
        """
        print("   -> [Controller] Processing Motion Signal...")
        
        if self.alarm.armed:
            self.alarm.sound_siren()
            # Maybe send a notification to phone here...
        elif self.is_night_mode:
            self.bulb.turn_on()
        else:
            print("   -> [Controller] Day time. Ignoring.")

#### CLIENT CODE

In [18]:
def main():
    # 1. Create Independent Devices
    # Notice: They don't take 'hub' in __init__
    front_door_sensor = MotionSensor("Front Door")
    living_room_light = SmartBulb("Living Room")
    security = AlarmSystem()

    # 2. Create Mediator (The Glue)
    # The mediator connects them.
    controller = HomeAutomationController(front_door_sensor, living_room_light, security)

    print("--- 1. Day Time ---")
    front_door_sensor.trigger() # Controller ignores

    print("\n--- 2. Night Time ---")
    controller.set_night_mode(True)
    front_door_sensor.trigger() # Controller turns on light

    print("\n--- 3. Armed & Dangerous ---")
    security.toggle_arm(True)
    front_door_sensor.trigger() # Controller sounds siren

if __name__ == "__main__":
    main()

--- 1. Day Time ---
[Front Door] Motion detected!
   -> [Controller] Processing Motion Signal...
   -> [Controller] Day time. Ignoring.

--- 2. Night Time ---
[Controller] Night Mode: True
[Front Door] Motion detected!
   -> [Controller] Processing Motion Signal...
[Living Room] ON

--- 3. Armed & Dangerous ---
[Alarm] Armed: True
[Front Door] Motion detected!
   -> [Controller] Processing Motion Signal...
[Alarm] 🚨 SIREN SOUNDING 🚨


Why the Pythonic Version is Better here

- **Reusability**: The Mot1ionSensor class in the Python version is completely generic. I can copy-paste that class into a Robotics project, and it works. In the Java version, MotionSensor expects a SmartHomeHub interface, so it's tightly coupled to the Smart Home project.
- **Observer/Callback Pattern**: Python's functions are first-class citizens. We don't need an interface to define a callback; we just pass the method self._handle_motion to the sensor.
- **Separation of Concerns**: The devices are pure hardware wrappers. The HomeAutomationController is pure logic.